In [1]:
import pandas as pd

In [2]:
train_df = pd.read_csv(r"D:\tensorflow\10_food_classes_10_percent\nlp_getting_started\train.csv")
test_df = pd.read_csv(r"D:\tensorflow\10_food_classes_10_percent\nlp_getting_started\test.csv")
train_df

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1


In [3]:
train_df_shuffled = train_df.sample(frac=1, random_state=42)
train_df_shuffled

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0
...,...,...,...,...,...
5226,7470,obliteration,Merica!,@Eganator2000 There aren't many Obliteration s...,0
5390,7691,panic,NaN,just had a panic attack bc I don't have enough...,0
860,1242,blood,NaN,Omron HEM-712C Automatic Blood Pressure Monito...,0
7603,10862,NaN,NaN,Officials say a quarantine is in place at an A...,1


In [4]:
test_df

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan
...,...,...,...,...
3258,10861,NaN,NaN,EARTHQUAKE SAFETY LOS ANGELES ÛÒ SAFETY FASTE...
3259,10865,NaN,NaN,Storm in RI worse than last hurricane. My city...
3260,10868,NaN,NaN,Green Line derailment in Chicago http://t.co/U...
3261,10874,NaN,NaN,MEG issues Hazardous Weather Outlook (HWO) htt...


In [5]:
train_df.target.value_counts()

target
0    4342
1    3271
Name: count, dtype: int64

In [6]:
print(f"Total training samples: {len(train_df)}")
print(f"Total test samples: {len(test_df)}")
print(f"Total samples: {len(train_df) + len(test_df)}")

Total training samples: 7613
Total test samples: 3263
Total samples: 10876


In [7]:
import random 
random_index = random.randint(0,len(train_df)-5)
for row in train_df_shuffled[['text','target']][random_index:random_index+5].itertuples():
    _,text,target = row
    print(f"Target:{target}","(real disater)" if target>0 else '(not real disaster)')
    print(f"Text:\n{text}\n")
    print("----\n")

Target:1 (real disater)
Text:
Christian Attacked by Muslims at the Temple Mount after Waving Israeli Flag via Pamela Geller - ... http://t.co/EMDJNNltP0

----

Target:0 (not real disaster)
Text:
My phone looks like it was in a car ship airplane accident. Terrible

----

Target:0 (not real disaster)
Text:
@nikistitz even the one that looked like a mudslide?

----

Target:0 (not real disaster)
Text:
The worst  voice I can ever hear is the 'Nikki your in trouble' voice from my mom

----

Target:0 (not real disaster)
Text:
Fatality scary af. Probably not the best falcon but still scary.

----



In [8]:
from sklearn.model_selection import train_test_split

train_sentences, val_sentences, train_labels, val_labels = train_test_split(train_df_shuffled["text"].to_numpy(),
                                                                            train_df_shuffled["target"].to_numpy(),
                                                                            test_size=0.1,
                                                                            random_state=42)

In [9]:
len(train_sentences), len(val_sentences), len(train_labels), len(val_labels)

(6851, 762, 6851, 762)

In [10]:
train_sentences[:10], train_labels[:10]

(array(['@mogacola @zamtriossu i screamed after hitting tweet',
        'Imagine getting flattened by Kurt Zouma',
        '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
        "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
        'Somehow find you and I collide http://t.co/Ee8RpOahPk',
        '@EvaHanderek @MarleyKnysh great times until the bus driver held us hostage in the mall parking lot lmfao',
        'destroy the free fandom honestly',
        'Weapons stolen from National Guard Armory in New Albany still missing #Gunsense http://t.co/lKNU8902JE',
        '@wfaaweather Pete when will the heat wave pass? Is it really going to be mid month? Frisco Boy Scouts have a canoe trip in Okla.',
        'Patient-reported outcomes in long-term survivors of metastatic colorectal cancer - British Journal of Surgery http://t.co/5Yl4DC1Tqt'],
       dtype=object),
 array([0,

In [11]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

In [12]:
text_vectorizer = TextVectorization(max_tokens=None,
                                   standardize="lower_and_strip_punctuation",
                                   split='whitespace',
                                   ngrams=None,
                                   output_mode='int',
                                   output_sequence_length=None)

In [13]:
text_vectorizer

In [14]:
round(sum([len(i.split()) for i in train_sentences])/len(train_sentences))

15

In [15]:
max_vocab_length = 1000
max_length = 15

text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode='int',
                                    output_sequence_length=max_length)

In [16]:
text_vectorizer

In [17]:
text_vectorizer.adapt(train_sentences)

In [18]:
sample_sentense = "I'm going to my home town"
text_vectorizer([sample_sentense])

<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[ 32, 104,   5,  13, 153, 801,   0,   0,   0,   0,   0,   0,   0,
          0,   0]], dtype=int64)>

In [19]:
random_sentense = random.choice(train_sentences)
print(f"Original text:\n{random_sentense}\
      \n\nVectorized version:")
text_vectorizer([random_sentense])

Original text:
MP trains derailment: Û÷ItÛªs the freakiest of freak accidentsÛª: 

MP trains derailment: Û÷ItÛªs the freakiest of ... http://t.co/uHXODSc7Wi      

Vectorized version:


<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[599, 757, 306,   1,   2,   1,   6,   1,   1, 599, 757, 306,   1,
          2,   1]], dtype=int64)>

In [20]:
words_in_vocab = text_vectorizer.get_vocabulary()
top_5_words = words_in_vocab[:5]
bottom_5_words = words_in_vocab[-5:]
print(f"Number of words in vocab:{len(words_in_vocab)}")
print(f"Top 5 most common words:{top_5_words}")
print(f"Bottom 5 least common words:{bottom_5_words}")

Number of words in vocab:1000
Top 5 most common words:['', '[UNK]', 'the', 'a', 'in']
Bottom 5 least common words:['reported', 'r', 'pray', 'playlist', 'patience']


In [21]:
tf.random.set_seed(42)

from tensorflow.keras import layers

embedding = layers.Embedding(input_dim=max_vocab_length,
                             output_dim=128,
                             embeddings_initializer="uniform",
                             input_length=max_length,
                             name="embedding_1")

In [22]:
random_sentense = random.choice(train_sentences)

print(f"Original Text:\n{random_sentense}\
      \n\n Embedded version:")

sample_embed = embedding(text_vectorizer([random_sentense]))
sample_embed

Original Text:
Christian Attacked by Muslims at the Temple Mount after Waving Israeli Flag via Pamela Geller - ... http://t.co/T1aa5Ov7Eg      

 Embedded version:


<tf.Tensor: shape=(1, 15, 128), dtype=float32, numpy=
array([[[-0.01184957, -0.00698131,  0.01856372, ..., -0.04042499,
         -0.02276573,  0.03803207],
        [ 0.00930488, -0.0267386 ,  0.01419879, ..., -0.025716  ,
         -0.03185933, -0.04095583],
        [-0.0117599 , -0.01091808, -0.01474231, ..., -0.04877792,
          0.04502607,  0.04560769],
        ...,
        [-0.03853712, -0.00808886, -0.00455149, ..., -0.01431368,
         -0.00335719,  0.03417077],
        [ 0.00032609,  0.03229712, -0.04722748, ...,  0.04806129,
         -0.04336817,  0.04661881],
        [-0.01674879, -0.02116917, -0.01771587, ...,  0.04282874,
          0.03964743, -0.02123314]]], dtype=float32)>

In [23]:
sample_embed[0][0]

<tf.Tensor: shape=(128,), dtype=float32, numpy=
array([-0.01184957, -0.00698131,  0.01856372, -0.01757951, -0.03611781,
        0.02593534, -0.00609197, -0.041349  ,  0.01075522, -0.00608165,
        0.03840644,  0.04466197,  0.03236271,  0.03317996, -0.03283732,
        0.04193619,  0.04846749, -0.03928413, -0.03765749, -0.02302333,
        0.04918578, -0.02466031, -0.03611602, -0.02074137,  0.02173131,
        0.01229777,  0.01621993, -0.02493103,  0.04888317, -0.01669313,
        0.00112926, -0.01696499,  0.00278055,  0.01996562,  0.0357842 ,
        0.01866045,  0.01015093,  0.02162338, -0.0315545 , -0.0192379 ,
        0.02549094,  0.0085826 ,  0.04737316, -0.04579332, -0.0329804 ,
        0.0467247 , -0.03837088,  0.01845255, -0.00146743,  0.02252059,
        0.04356681,  0.0278376 , -0.02842869,  0.01546023, -0.02011278,
       -0.03762025, -0.02479731,  0.04763193, -0.02532642,  0.01889427,
       -0.02525011,  0.04183521, -0.00040603, -0.03198971,  0.02626504,
        0.038568

***Naive Bayes(baseline)***

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

model_0 =Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])

model_0.fit(train_sentences, train_labels)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

In [25]:
baseline_score = model_0.score(val_sentences,val_labels)
print(baseline_score)
print(f"Our baseline model achieves an accuracy of: {baseline_score*100:.2f}%")

0.7926509186351706
Our baseline model achieves an accuracy of: 79.27%


In [26]:
baseline_pred = model_0.predict(val_sentences)
baseline_pred[:20]

array([1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1],
      dtype=int64)

In [27]:
from sklearn.metrics import accuracy_score,precision_recall_fscore_support

def calculate_results(y_true, y_pred):
    model_accuracy = accuracy_score(y_true, y_pred) * 100
    model_precision, model_recall, model_f1,_ = precision_recall_fscore_support(y_true,y_pred, average='weighted')
    model_results = {"accuracy": model_accuracy,
                     "precision": model_precision,
                     "recall": model_recall,
                     "f1": model_f1}
    return model_results

In [28]:
baseline_results = calculate_results(y_true=val_labels,y_pred=baseline_pred)
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1': 0.7862189758049549}

In [29]:
from helper_functions import create_tensorboard_callback
SAVE_DIR = "model_logs"

In [30]:
from tensorflow.keras import layers
inputs =layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = embedding(x)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_1 = tf.keras.Model(inputs,outputs,name="model_1_dense")


In [31]:
model_1.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [32]:
model_1.summary()

Model: "model_1_dense"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (TextV  (None, 15)               0         
 ectorization)                                                   
                                                                 
 embedding_1 (Embedding)     (None, 15, 128)           128000    
                                                                 
 global_average_pooling1d (G  (None, 128)              0         
 lobalAveragePooling1D)                                          
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 128,129
Trainable params: 128,129
Non-t

In [33]:
# Fit the model
model_1_history = model_1.fit(train_sentences, # input sentences can be a list of strings due to text preprocessing layer built-in model
                              train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels),
                              callbacks=[create_tensorboard_callback(dir_name=SAVE_DIR, 
                                                                     experiment_name="simple_dense_model")])

Saving TensorBoard log files to: model_logs/simple_dense_model/20260917-020148
Epoch 1/5
215/215 [==============================] - 5s 7ms/step - loss: 0.6203 - accuracy: 0.6675 - val_loss: 0.5614 - val_accuracy: 0.7283
Epoch 2/5
215/215 [==============================] - 1s 6ms/step - loss: 0.4928 - accuracy: 0.7834 - val_loss: 0.5007 - val_accuracy: 0.7598
Epoch 3/5
215/215 [==============================] - 1s 6ms/step - loss: 0.4371 - accuracy: 0.8059 - val_loss: 0.4866 - val_accuracy: 0.7664
Epoch 4/5
215/215 [==============================] - 1s 6ms/step - loss: 0.4115 - accuracy: 0.8181 - val_loss: 0.4856 - val_accuracy: 0.7782
Epoch 5/5
215/215 [==============================] - 1s 6ms/step - loss: 0.3946 - accuracy: 0.8243 - val_loss: 0.4887 - val_accuracy: 0.7769


In [34]:
model_1.evaluate(val_sentences, val_labels)

24/24 [==============================] - 0s 3ms/step - loss: 0.4887 - accuracy: 0.7769


[0.48867344856262207, 0.7769029140472412]

In [35]:
embedding.weights

[<tf.Variable 'embedding_1/embeddings:0' shape=(1000, 128) dtype=float32, numpy=
 array([[ 0.01528927, -0.01761752, -0.02377709, ...,  0.02270182,
          0.07185008, -0.00211658],
        [-0.00793551, -0.00797752, -0.0052463 , ...,  0.02818114,
          0.05144301, -0.00872088],
        [ 0.01657409,  0.0647485 ,  0.03864899, ..., -0.03982078,
          0.05776957,  0.05248051],
        ...,
        [ 0.00578395,  0.00494304,  0.01596744, ..., -0.03112444,
         -0.01295553, -0.00595034],
        [ 0.08857379,  0.07654488,  0.11425049, ..., -0.06348468,
          0.02244311,  0.07026725],
        [-0.03430416, -0.05291138,  0.02992735, ...,  0.00238389,
         -0.01335472,  0.0288819 ]], dtype=float32)>]

In [36]:
embed_weights = model_1.get_layer("embedding_1").get_weights()[0]
print(embed_weights.shape)

(1000, 128)


In [37]:
model_1_pred_probs = model_1.predict(val_sentences)
model_1_pred_probs[:10]

24/24 [==============================] - 0s 2ms/step


array([[0.5322457 ],
       [0.68983495],
       [0.96992326],
       [0.10638279],
       [0.17003012],
       [0.9242093 ],
       [0.85960734],
       [0.8313793 ],
       [0.7129582 ],
       [0.16848624]], dtype=float32)

In [38]:
model_1_preds = tf.squeeze(tf.round(model_1_pred_probs)) 
model_1_preds[:20]

<tf.Tensor: shape=(20,), dtype=float32, numpy=
array([1., 1., 1., 0., 0., 1., 1., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 0.], dtype=float32)>

In [39]:
model_1_results = calculate_results(y_true=val_labels, 
                                    y_pred=model_1_preds)
model_1_results

{'accuracy': 77.69028871391076,
 'precision': 0.7813687007758338,
 'recall': 0.7769028871391076,
 'f1': 0.7737651515426378}

In [40]:
import numpy as np
np.array(list(model_1_results.values())) > np.array(list(baseline_results.values()))

array([False, False, False, False])

In [41]:
def compare_baseline_to_new_results(baseline_results,new_model_results):
    for key,value in baseline_results.items():
        print(f"Baseline {key}: {value:.2f}, New {key}: {new_model_results[key]:.2f}, Difference: {new_model_results[key]- value:.2f}")

compare_baseline_to_new_results(baseline_results=baseline_results,new_model_results=model_1_results)

Baseline accuracy: 79.27, New accuracy: 77.69, Difference: -1.57
Baseline precision: 0.81, New precision: 0.78, Difference: -0.03
Baseline recall: 0.79, New recall: 0.78, Difference: -0.02
Baseline f1: 0.79, New f1: 0.77, Difference: -0.01


In [42]:
words_in_vocab = text_vectorizer.get_vocabulary()
len(words_in_vocab), words_in_vocab[:10]

(1000, ['', '[UNK]', 'the', 'a', 'in', 'to', 'of', 'and', 'i', 'is'])

In [43]:
model_1.summary()

Model: "model_1_dense"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (TextV  (None, 15)               0         
 ectorization)                                                   
                                                                 
 embedding_1 (Embedding)     (None, 15, 128)           128000    
                                                                 
 global_average_pooling1d (G  (None, 128)              0         
 lobalAveragePooling1D)                                          
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 128,129
Trainable params: 128,129
Non-t

In [44]:
embed_weights = model_1.get_layer("embedding_1").get_weights()[0]
print(embed_weights.shape)

(1000, 128)


***2.LSTM***

In [45]:
tf.random.set_seed(42)

from tensorflow.keras import layers

model_2_embedding = layers.Embedding(input_dim=max_vocab_length,
                                     output_dim=128,
                                     embeddings_initializer="uniform",
                                     input_length=max_length,
                                     name="embedding_2")

#Create LSTM

inputs = layers.Input(shape=(1,), dtype="string")
x=text_vectorizer(inputs)
x=model_2_embedding(x)
print(x.shape)

x = layers.LSTM(64)(x)
print(x.shape)

outputs = layers.Dense(1, activation='sigmoid')(x)

model_2 = tf.keras.Model(inputs,outputs,name='model_2_LSTM')

(None, 15, 128)
(None, 64)


In [46]:
model_2.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [47]:
model_2.summary()

Model: "model_2_LSTM"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (TextV  (None, 15)               0         
 ectorization)                                                   
                                                                 
 embedding_2 (Embedding)     (None, 15, 128)           128000    
                                                                 
 lstm (LSTM)                 (None, 64)                49408     
                                                                 
 dense_1 (Dense)             (None, 1)                 65        
                                                                 
Total params: 177,473
Trainable params: 177,473
Non-trainable params: 0
________________________________________________

In [48]:
# Fit model
model_2_history = model_2.fit(train_sentences,
                              train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels),
                              callbacks=[create_tensorboard_callback(SAVE_DIR, 
                                                                     "LSTM")])

Saving TensorBoard log files to: model_logs/LSTM/20260917-020158
Epoch 1/5
215/215 [==============================] - 7s 12ms/step - loss: 0.5256 - accuracy: 0.7364 - val_loss: 0.4864 - val_accuracy: 0.7651
Epoch 2/5
215/215 [==============================] - 2s 9ms/step - loss: 0.4210 - accuracy: 0.8136 - val_loss: 0.4764 - val_accuracy: 0.7795
Epoch 3/5
215/215 [==============================] - 2s 9ms/step - loss: 0.3932 - accuracy: 0.8285 - val_loss: 0.4970 - val_accuracy: 0.7703
Epoch 4/5
215/215 [==============================] - 2s 11ms/step - loss: 0.3711 - accuracy: 0.8402 - val_loss: 0.5088 - val_accuracy: 0.7756
Epoch 5/5
215/215 [==============================] - 2s 11ms/step - loss: 0.3469 - accuracy: 0.8542 - val_loss: 0.5219 - val_accuracy: 0.7677


In [49]:
model_2_pred_probs = model_2.predict(val_sentences)
model_2_pred_probs.shape, model_2_pred_probs[:10]

24/24 [==============================] - 0s 4ms/step


((762, 1),
 array([[0.49997854],
        [0.6586019 ],
        [0.9686817 ],
        [0.0858846 ],
        [0.08921135],
        [0.9938305 ],
        [0.86306006],
        [0.92679113],
        [0.8292705 ],
        [0.10882235]], dtype=float32))

In [50]:
model_2_preds = tf.squeeze(tf.round(model_2_pred_probs))
model_2_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0.], dtype=float32)>

In [51]:
# Calculate LSTM model results
model_2_results = calculate_results(y_true=val_labels,
                                    y_pred=model_2_preds)
model_2_results

{'accuracy': 76.77165354330708,
 'precision': 0.7702262582836057,
 'recall': 0.7677165354330708,
 'f1': 0.7651303822545579}

In [52]:
compare_baseline_to_new_results(baseline_results, model_2_results)

Baseline accuracy: 79.27, New accuracy: 76.77, Difference: -2.49
Baseline precision: 0.81, New precision: 0.77, Difference: -0.04
Baseline recall: 0.79, New recall: 0.77, Difference: -0.02
Baseline f1: 0.79, New f1: 0.77, Difference: -0.02


***3.GRU***

In [53]:
tf.random.set_seed(42)

model_3_embedding = layers.Embedding(input_dim=max_vocab_length,
                                     output_dim=128,
                                     embeddings_initializer='uniform',
                                     input_length=max_length,
                                     name='embedding_3')

inputs = layers.Input(shape=(1,), dtype='string')
x= text_vectorizer(inputs)
x= model_3_embedding(x)
x=layers.GRU(64)(x)

outputs = layers.Dense(1,activation='sigmoid')(x)

model_3 = tf.keras.Model(inputs,outputs,name='model_3_GRU')

In [54]:
model_3.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [55]:
model_3.summary()

Model: "model_3_GRU"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (TextV  (None, 15)               0         
 ectorization)                                                   
                                                                 
 embedding_3 (Embedding)     (None, 15, 128)           128000    
                                                                 
 gru (GRU)                   (None, 64)                37248     
                                                                 
 dense_2 (Dense)             (None, 1)                 65        
                                                                 
Total params: 165,313
Trainable params: 165,313
Non-trainable params: 0
_________________________________________________

In [56]:
# Fit model
model_3_history = model_3.fit(train_sentences,
                              train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels),
                              callbacks=[create_tensorboard_callback(SAVE_DIR, "GRU")])

Saving TensorBoard log files to: model_logs/GRU/20260917-020215
Epoch 1/5
215/215 [==============================] - 4s 14ms/step - loss: 0.5421 - accuracy: 0.7101 - val_loss: 0.4827 - val_accuracy: 0.7677
Epoch 2/5
215/215 [==============================] - 2s 11ms/step - loss: 0.4221 - accuracy: 0.8114 - val_loss: 0.4822 - val_accuracy: 0.7782
Epoch 3/5
215/215 [==============================] - 2s 10ms/step - loss: 0.3917 - accuracy: 0.8283 - val_loss: 0.4998 - val_accuracy: 0.7730
Epoch 4/5
215/215 [==============================] - 2s 10ms/step - loss: 0.3681 - accuracy: 0.8415 - val_loss: 0.5141 - val_accuracy: 0.7612
Epoch 5/5
215/215 [==============================] - 2s 11ms/step - loss: 0.3474 - accuracy: 0.8529 - val_loss: 0.5186 - val_accuracy: 0.7664


In [57]:
model_3_pred_probs = model_3.predict(val_sentences)
model_3_pred_probs.shape, model_3_pred_probs[:10]

24/24 [==============================] - 0s 5ms/step


((762, 1),
 array([[0.68187827],
        [0.6219189 ],
        [0.95817626],
        [0.08612284],
        [0.06357006],
        [0.9782819 ],
        [0.86036915],
        [0.94829595],
        [0.7513366 ],
        [0.1035511 ]], dtype=float32))

In [58]:
model_3_preds = tf.squeeze(tf.round(model_3_pred_probs))
model_3_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([1., 1., 1., 0., 0., 1., 1., 1., 1., 0.], dtype=float32)>

In [59]:
model_3_results = calculate_results(y_true=val_labels, 
                                    y_pred=model_3_preds)
model_3_results

{'accuracy': 76.64041994750657,
 'precision': 0.7681630705252752,
 'recall': 0.7664041994750657,
 'f1': 0.764152388105216}

In [60]:
compare_baseline_to_new_results(baseline_results, model_3_results)

Baseline accuracy: 79.27, New accuracy: 76.64, Difference: -2.62
Baseline precision: 0.81, New precision: 0.77, Difference: -0.04
Baseline recall: 0.79, New recall: 0.77, Difference: -0.03
Baseline f1: 0.79, New f1: 0.76, Difference: -0.02


***4.Bidirectional***

In [61]:
tf.random.set_seed(42)

model_4_embedding = tf.keras.layers.Embedding(input_dim=max_vocab_length,
                                              output_dim=128,
                                              embeddings_initializer='uniform',
                                              input_length=max_length,
                                              name='embedding_4')

inputs = tf.keras.layers.Input(shape=(1,),dtype='string')
x=text_vectorizer(inputs)
x=model_4_embedding(x)
x=tf.keras.layers.Bidirectional(layers.LSTM(64))(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model_4 = tf.keras.Model(inputs,outputs,name='model_4_Bidirectional')

In [62]:
model_4.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [63]:
model_4.summary()

Model: "model_4_Bidirectional"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (TextV  (None, 15)               0         
 ectorization)                                                   
                                                                 
 embedding_4 (Embedding)     (None, 15, 128)           128000    
                                                                 
 bidirectional (Bidirectiona  (None, 128)              98816     
 l)                                                              
                                                                 
 dense_3 (Dense)             (None, 1)                 129       
                                                                 
Total params: 226,945
Trainable params: 226,9

In [64]:
model_4_history = model_4.fit(train_sentences,
                              train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels),
                              callbacks=[create_tensorboard_callback(SAVE_DIR, "bidirectional_RNN")])

Saving TensorBoard log files to: model_logs/bidirectional_RNN/20260917-020230
Epoch 1/5
215/215 [==============================] - 7s 20ms/step - loss: 0.5234 - accuracy: 0.7386 - val_loss: 0.4840 - val_accuracy: 0.7677
Epoch 2/5
215/215 [==============================] - 3s 15ms/step - loss: 0.4169 - accuracy: 0.8137 - val_loss: 0.4850 - val_accuracy: 0.7782
Epoch 3/5
215/215 [==============================] - 3s 15ms/step - loss: 0.3851 - accuracy: 0.8333 - val_loss: 0.5035 - val_accuracy: 0.7690
Epoch 4/5
215/215 [==============================] - 3s 13ms/step - loss: 0.3571 - accuracy: 0.8483 - val_loss: 0.5264 - val_accuracy: 0.7612
Epoch 5/5
215/215 [==============================] - 3s 14ms/step - loss: 0.3301 - accuracy: 0.8631 - val_loss: 0.5469 - val_accuracy: 0.7480


In [65]:
model_4_pred_probs = model_4.predict(val_sentences)
model_4_pred_probs[:10]

24/24 [==============================] - 1s 5ms/step


array([[0.7506622 ],
       [0.6602833 ],
       [0.96169996],
       [0.09062905],
       [0.14316359],
       [0.99218327],
       [0.63044286],
       [0.97306305],
       [0.88383657],
       [0.12051395]], dtype=float32)

In [66]:
model_4_preds = tf.squeeze(tf.round(model_4_pred_probs))
model_4_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([1., 1., 1., 0., 0., 1., 1., 1., 1., 0.], dtype=float32)>

In [67]:
model_4_results = calculate_results(val_labels, model_4_preds)
model_4_results

{'accuracy': 74.80314960629921,
 'precision': 0.7483325878156181,
 'recall': 0.7480314960629921,
 'f1': 0.7462839871172066}

In [68]:
compare_baseline_to_new_results(baseline_results, model_4_results)

Baseline accuracy: 79.27, New accuracy: 74.80, Difference: -4.46
Baseline precision: 0.81, New precision: 0.75, Difference: -0.06
Baseline recall: 0.79, New recall: 0.75, Difference: -0.04
Baseline f1: 0.79, New f1: 0.75, Difference: -0.04
